# 267. Palindrome Permutation II
https://leetcode.com/problems/palindrome-permutation-ii/

## Complexity Comparison

| Approach | Time | Space | Description |
|----------|------|-------|-------------|
| Generate all permutations | O(n!) | O(n!) | Generate all, filter palindromes |
| **Half-string backtrack (Optimal)** | **O((n/2)!)** | **O(n)** | Build half, mirror for palindrome |

## Methodology

First check if a palindrome permutation is possible (at most one odd-frequency char). Then build only the first half of the palindrome using backtracking with the half-counts of each character. The center character (if string length is odd) is placed in the middle. Mirror the first half to complete each palindrome. This avoids generating duplicate permutations.

## Solutions

### C#

In [ ]:
public class Solution {
    public IList<string> GeneratePalindromes(string s) {
        var result = new List<string>();
        var count = new Dictionary<char, int>();
        foreach (char c in s) count[c] = count.GetValueOrDefault(c) + 1;
        string mid = "";
        var half = new List<char>();
        foreach (var kv in count) {
            if (kv.Value % 2 == 1) {
                if (mid.Length > 0) return result;
                mid = kv.Key.ToString();
            }
            for (int i = 0; i < kv.Value / 2; i++) half.Add(kv.Key);
        }
        half.Sort();
        bool[] used = new bool[half.Count];
        Backtrack(half, used, new char[half.Count], 0, mid, result);
        return result;
    }
    
    private void Backtrack(List<char> half, bool[] used, char[] cur, int idx, string mid, List<string> result) {
        if (idx == half.Count) {
            string h = new string(cur);
            char[] rev = cur.Reverse().ToArray();
            result.Add(h + mid + new string(rev));
            return;
        }
        for (int i = 0; i < half.Count; i++) {
            if (used[i] || (i > 0 && half[i] == half[i-1] && !used[i-1])) continue;
            used[i] = true;
            cur[idx] = half[i];
            Backtrack(half, used, cur, idx + 1, mid, result);
            used[i] = false;
        }
    }
}

### Python

In [ ]:
from collections import Counter

class Solution:
    def generatePalindromes(self, s: str) -> list[str]:
        count = Counter(s)
        odd_chars = [c for c, v in count.items() if v % 2 == 1]
        if len(odd_chars) > 1:
            return []
        mid = odd_chars[0] if odd_chars else ''
        half = []
        for c, v in sorted(count.items()):
            half.extend([c] * (v // 2))
        result = []
        def backtrack(path, remaining):
            if not remaining:
                h = ''.join(path)
                result.append(h + mid + h[::-1])
                return
            prev = None
            for i, c in enumerate(remaining):
                if c == prev:
                    continue
                prev = c
                backtrack(path + [c], remaining[:i] + remaining[i+1:])
        backtrack([], half)
        return result

### Go

In [ ]:
func generatePalindromes(s string) []string {
    count := make(map[byte]int)
    for i := range s {
        count[s[i]]++
    }
    mid := ""
    var half []byte
    for c, v := range count {
        if v%2 == 1 {
            if mid != "" {
                return nil
            }
            mid = string(c)
        }
        for i := 0; i < v/2; i++ {
            half = append(half, c)
        }
    }
    sort.Slice(half, func(i, j int) bool { return half[i] < half[j] })
    var result []string
    used := make([]bool, len(half))
    var backtrack func(cur []byte)
    backtrack = func(cur []byte) {
        if len(cur) == len(half) {
            rev := make([]byte, len(cur))
            for i, b := range cur {
                rev[len(cur)-1-i] = b
            }
            result = append(result, string(cur)+mid+string(rev))
            return
        }
        for i := range half {
            if used[i] || (i > 0 && half[i] == half[i-1] && !used[i-1]) {
                continue
            }
            used[i] = true
            backtrack(append(cur, half[i]))
            cur = cur[:len(cur)-1]
            used[i] = false
        }
    }
    backtrack(nil)
    return result
}

### Rust

In [ ]:
use std::collections::HashMap;

impl Solution {
    pub fn generate_palindromes(s: String) -> Vec<String> {
        let mut count = HashMap::new();
        for c in s.chars() {
            *count.entry(c).or_insert(0) += 1;
        }
        let mut mid = String::new();
        let mut half: Vec<char> = Vec::new();
        for (&c, &v) in &count {
            if v % 2 == 1 {
                if !mid.is_empty() { return vec![]; }
                mid.push(c);
            }
            for _ in 0..v / 2 {
                half.push(c);
            }
        }
        half.sort();
        let mut result = Vec::new();
        let mut used = vec![false; half.len()];
        let mut cur = Vec::new();
        Self::backtrack(&half, &mut used, &mut cur, &mid, &mut result);
        result
    }
    
    fn backtrack(half: &[char], used: &mut Vec<bool>, cur: &mut Vec<char>,
                 mid: &str, result: &mut Vec<String>) {
        if cur.len() == half.len() {
            let h: String = cur.iter().collect();
            let rev: String = cur.iter().rev().collect();
            result.push(format!("{}{}{}", h, mid, rev));
            return;
        }
        for i in 0..half.len() {
            if used[i] || (i > 0 && half[i] == half[i-1] && !used[i-1]) {
                continue;
            }
            used[i] = true;
            cur.push(half[i]);
            Self::backtrack(half, used, cur, mid, result);
            cur.pop();
            used[i] = false;
        }
    }
}

## Example Scenarios

1. **`s = "aabb"`** - Returns `["abba", "baab"]`. Two distinct palindromic permutations.

2. **`s = "abc"`** - Three odd-count characters. No palindrome possible. Returns `[]`.

3. **`s = "aab"`** - Half = "a", mid = "b". Returns `["aba"]`.

4. **`s = "a"`** - Single character. Returns `["a"]`.

5. **`s = "aabbcc"`** - Three pairs, even length. Multiple palindromes like "abccba", "acbcba", etc.

![image](attachment:image.png)